# Session 10: Convolutional Neural Networks

**CVI4IC - Summer Semester 2026**

Building and training CNNs for image classification.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Building a CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 2. Data & Training

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,
                                transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False,
                               transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f}")

## 3. Visualizing Learned Filters

In [ ]:
# Visualize first layer filters
filters = model.features[0].weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("Learned Conv1 Filters")
plt.tight_layout()
plt.show()

## 4. Transfer Learning with ResNet

In [ ]:
# Load a pretrained ResNet (for demonstration)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze feature layers
for param in resnet.parameters():
    param.requires_grad = False

# Replace final FC layer for our task
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 10)

print(f"ResNet18 final layer: {resnet.fc}")
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

## Exercises

1. Add more convolutional layers to `SimpleCNN` and compare test accuracy
2. Train on CIFAR-10 instead of MNIST (adjust input channels to 3)
3. Fine-tune a pretrained ResNet on a small custom dataset

In [ ]:
# Your code here
